# 🔐 DIFFIE-HELLMAN

**Protocolo de Intercambio de Claves**

---

## Estructura del Notebook

1. **PARTE PRÁCTICA** - Ejecuta y experimenta con el protocolo
2. **PARTE TEÓRICA** - Fundamentos matemáticos y explicación detallada

---

**Objetivo:** Permitir a dos partes (Alice y Bob) establecer un secreto compartido sobre un canal público inseguro.

**Inventores:** Whitfield Diffie y Martin Hellman (1976)

**Seguridad:** Basada en el Problema del Logaritmo Discreto

---

# 📝 PARTE PRÁCTICA

## 🎯 Ejemplo Completo: Alice y Bob establecen un secreto compartido

### ⚙️ Configuración del Ejemplo

Modifica estos valores para experimentar:

In [ ]:
import random
from math import gcd

# ============================================
# CONFIGURACIÓN - Modifica aquí
# ============================================

# Parámetros públicos (conocidos por todos)
p = 23      # Número primo
g = 5       # Generador

# Claves privadas (SECRETAS)
a = 6       # Clave privada de Alice
b = 15      # Clave privada de Bob

print("CONFIGURACIÓN INICIAL")
print("="*60)
print(f"\nParámetros públicos:")
print(f"  p (primo)     = {p}")
print(f"  g (generador) = {g}")
print(f"\nClaves privadas:")
print(f"  a (Alice) = {a}  [SECRETA]")
print(f"  b (Bob)   = {b}  [SECRETA]")

### 1. Función: Exponenciación Modular Rápida

In [ ]:
def exponenciacion_rapida(base, exponente, modulo):
    """
    Calcula (base^exponente) mod modulo eficientemente.
    
    Algoritmo de exponenciación modular rápida (square-and-multiply):
    1. Representa el exponente en binario
    2. Para cada bit (de derecha a izquierda):
       - Si el bit es 1: multiplica el resultado por la base actual
       - Eleva la base al cuadrado
    3. Aplica módulo en cada paso para mantener números manejables
    
    Ejemplo: 5^6 mod 23
        6 en binario = 110
        
        Bit 0 (derecha): 0 → solo elevar al cuadrado
          base = 5, resultado = 1
          base = 5^2 = 25 ≡ 2 (mod 23)
        
        Bit 1: 1 → multiplicar y elevar al cuadrado
          resultado = 1 × 2 = 2
          base = 2^2 = 4
        
        Bit 2: 1 → multiplicar
          resultado = 2 × 4 = 8
        
        Resultado: 5^6 mod 23 = 8
    
    Parámetros:
        base: base de la exponenciación
        exponente: exponente
        modulo: módulo
    
    Retorna:
        (base^exponente) mod modulo
    """
    resultado = 1
    base = base % modulo
    
    while exponente > 0:
        # Si el exponente es impar (bit menos significativo es 1)
        if exponente % 2 == 1:
            resultado = (resultado * base) % modulo
        
        # Siguiente bit (dividir exponente entre 2)
        exponente = exponente >> 1
        # Elevar base al cuadrado
        base = (base * base) % modulo
    
    return resultado

# Prueba de la función
print("Pruebas de exponenciación modular rápida:")
print("="*60)
pruebas = [
    (5, 6, 23),
    (5, 15, 23),
    (7, 5, 13),
]

for base_t, exp_t, mod_t in pruebas:
    resultado = exponenciacion_rapida(base_t, exp_t, mod_t)
    # Verificar con pow de Python
    verificacion = pow(base_t, exp_t, mod_t)
    print(f"{base_t}^{exp_t} mod {mod_t} = {resultado}  ✓" if resultado == verificacion else "✗")

### 2. Función: Calcular Clave Pública

In [ ]:
def calcular_clave_publica(g, clave_privada, p):
    """
    Calcula la clave pública a partir de la clave privada.
    
    Fórmula: clave_publica = g^clave_privada mod p
    
    Proceso:
    1. Toma el generador g
    2. Elévalo a la potencia de la clave privada
    3. Aplica módulo p
    
    Esta operación es fácil de calcular (dirección forward),
    pero dado g, clave_publica y p, es computacionalmente
    difícil encontrar clave_privada (Problema del Logaritmo Discreto).
    
    Parámetros:
        g: generador (parámetro público)
        clave_privada: clave privada del usuario
        p: primo (parámetro público)
    
    Retorna:
        Clave pública
    """
    return exponenciacion_rapida(g, clave_privada, p)

# Calcular claves públicas de Alice y Bob
print("PASO 1: CALCULAR CLAVES PÚBLICAS")
print("="*60)

# Alice calcula su clave pública
A = calcular_clave_publica(g, a, p)
print(f"\nAlice calcula:")
print(f"  A = g^a mod p")
print(f"  A = {g}^{a} mod {p}")
print(f"  A = {A}")

# Bob calcula su clave pública
B = calcular_clave_publica(g, b, p)
print(f"\nBob calcula:")
print(f"  B = g^b mod p")
print(f"  B = {g}^{b} mod {p}")
print(f"  B = {B}")

print(f"\n📢 Alice y Bob intercambian sus claves públicas:")
print(f"   Alice envía a Bob: A = {A}")
print(f"   Bob envía a Alice: B = {B}")
print(f"\n⚠️ Un atacante puede ver A y B, ¡pero no a ni b!")

### 3. Función: Calcular Secreto Compartido

In [ ]:
def calcular_secreto_compartido(clave_publica_otro, clave_privada_propia, p):
    """
    Calcula el secreto compartido.
    
    Fórmula: secreto = (clave_publica_otro)^clave_privada_propia mod p
    
    La magia de Diffie-Hellman:
    - Alice calcula: B^a mod p = (g^b)^a mod p = g^(ba) mod p
    - Bob calcula:   A^b mod p = (g^a)^b mod p = g^(ab) mod p
    
    Como ba = ab, ambos obtienen el mismo resultado: g^(ab) mod p
    
    Esto funciona por la propiedad conmutativa de la multiplicación:
    (g^a)^b = g^(a×b) = g^(b×a) = (g^b)^a
    
    Parámetros:
        clave_publica_otro: clave pública de la otra parte
        clave_privada_propia: clave privada propia
        p: primo (parámetro público)
    
    Retorna:
        Secreto compartido
    """
    return exponenciacion_rapida(clave_publica_otro, clave_privada_propia, p)

# Alice y Bob calculan el secreto compartido
print("PASO 2: CALCULAR SECRETO COMPARTIDO")
print("="*60)

# Alice usa la clave pública de Bob y su clave privada
secreto_alice = calcular_secreto_compartido(B, a, p)
print(f"\nAlice calcula:")
print(f"  secreto = B^a mod p")
print(f"  secreto = {B}^{a} mod {p}")
print(f"  secreto = {secreto_alice}")
print(f"\n  Expandiendo: B = g^b = {g}^{b}")
print(f"  Entonces: ({g}^{b})^{a} = {g}^({b}×{a}) = {g}^{a*b} mod {p} = {secreto_alice}")

# Bob usa la clave pública de Alice y su clave privada
secreto_bob = calcular_secreto_compartido(A, b, p)
print(f"\nBob calcula:")
print(f"  secreto = A^b mod p")
print(f"  secreto = {A}^{b} mod {p}")
print(f"  secreto = {secreto_bob}")
print(f"\n  Expandiendo: A = g^a = {g}^{a}")
print(f"  Entonces: ({g}^{a})^{b} = {g}^({a}×{b}) = {g}^{a*b} mod {p} = {secreto_bob}")

### 4. Verificación Final

In [ ]:
print("="*60)
print("VERIFICACIÓN FINAL")
print("="*60)

print(f"\nSecreto de Alice: {secreto_alice}")
print(f"Secreto de Bob:   {secreto_bob}")

if secreto_alice == secreto_bob:
    print(f"\n✅ ¡ÉXITO! Ambos tienen el mismo secreto compartido: {secreto_alice}")
    print(f"\nAmbos calcularon: g^(ab) mod p = {g}^({a}×{b}) mod {p} = {g}^{a*b} mod {p}")
    
    # Verificar directamente
    verificacion = exponenciacion_rapida(g, a*b, p)
    print(f"Verificación directa: {g}^{a*b} mod {p} = {verificacion}")
else:
    print(f"\n❌ ERROR: Los secretos no coinciden")

print(f"\n{'='*60}")
print("RESUMEN DEL PROTOCOLO")
print("="*60)
print(f"\n1. Parámetros públicos: p={p}, g={g}")
print(f"2. Alice: privada={a}, pública={A}")
print(f"3. Bob:   privada={b}, pública={B}")
print(f"4. Intercambio: Alice↔Bob envían sus claves públicas")
print(f"5. Secreto compartido: {secreto_alice}")
print(f"\n💡 Ahora Alice y Bob pueden usar este secreto como clave de cifrado simétrico")

### 📊 Visualización del Protocolo

In [ ]:
def visualizar_protocolo(p, g, a, A, b, B, secreto):
    """Crea una visualización ASCII del protocolo."""
    
    print("\n" + "="*80)
    print(" "*25 + "DIAGRAMA DEL PROTOCOLO")
    print("="*80)
    
    print(f"""
    ALICE                                                   BOB
    ┌─────────┐                                         ┌─────────┐
    │ Privada │                                         │ Privada │
    │ a = {a:<3} │                                         │ b = {b:<3} │
    └─────────┘                                         └─────────┘
         │                                                   │
         │ A = g^a mod p                     B = g^b mod p │
         │ A = {g}^{a} mod {p}                    B = {g}^{b} mod {p} │
         ▼ A = {A:<3}                                B = {B:<3} ▼
    ┌─────────┐                                         ┌─────────┐
    │ Pública │                                         │ Pública │
    │ A = {A:<3} │─────────── Intercambio ───────────▶│ B = {B:<3} │
    │         │◀────────── Público (A, B) ────────────│         │
    └─────────┘                                         └─────────┘
         │                                                   │
         │ s = B^a mod p                     s = A^b mod p │
         │ s = {B}^{a} mod {p}                    s = {A}^{b} mod {p} │
         ▼ s = {secreto:<3}                              s = {secreto:<3} ▼
    ┌─────────┐                                         ┌─────────┐
    │ Secreto │                                         │ Secreto │
    │ s = {secreto:<3} │◀──── ¡Mismo valor! ─────────▶│ s = {secreto:<3} │
    └─────────┘                                         └─────────┘
    """)
    
    print("="*80)
    print("\n⚠️ Un atacante (Eve) que intercepta el canal ve:")
    print(f"   • p = {p}")
    print(f"   • g = {g}")
    print(f"   • A = {A}")
    print(f"   • B = {B}")
    print(f"\n   Pero NO puede calcular:")
    print(f"   • a = {a} (clave privada de Alice)")
    print(f"   • b = {b} (clave privada de Bob)")
    print(f"   • secreto = {secreto}")
    print(f"\n   Esto es el Problema del Logaritmo Discreto 🔒")

visualizar_protocolo(p, g, a, A, b, B, secreto_alice)

---

## 🧪 Casos de Uso para Probar

### Caso 1: Ejemplo Simple

In [ ]:
print("CASO 1: p=23, g=5, a=6, b=15")
print("-"*60)

p1, g1, a1, b1 = 23, 5, 6, 15
A1 = calcular_clave_publica(g1, a1, p1)
B1 = calcular_clave_publica(g1, b1, p1)
s_alice1 = calcular_secreto_compartido(B1, a1, p1)
s_bob1 = calcular_secreto_compartido(A1, b1, p1)

print(f"Claves públicas: A={A1}, B={B1}")
print(f"Secreto de Alice: {s_alice1}")
print(f"Secreto de Bob:   {s_bob1}")
print(f"✓ Coinciden: {s_alice1 == s_bob1}\n")

### Caso 2: Otro Ejemplo

In [ ]:
print("CASO 2: p=47, g=7, a=12, b=25")
print("-"*60)

p2, g2, a2, b2 = 47, 7, 12, 25
A2 = calcular_clave_publica(g2, a2, p2)
B2 = calcular_clave_publica(g2, b2, p2)
s_alice2 = calcular_secreto_compartido(B2, a2, p2)
s_bob2 = calcular_secreto_compartido(A2, b2, p2)

print(f"Claves públicas: A={A2}, B={B2}")
print(f"Secreto de Alice: {s_alice2}")
print(f"Secreto de Bob:   {s_bob2}")
print(f"✓ Coinciden: {s_alice2 == s_bob2}\n")

### Caso 3: Con Números más Grandes

In [ ]:
print("CASO 3: Números más grandes")
print("-"*60)

p3, g3 = 997, 7  # Primo más grande
a3 = random.randint(100, 500)
b3 = random.randint(100, 500)

print(f"Parámetros: p={p3}, g={g3}")
print(f"Claves privadas: a={a3}, b={b3}")

A3 = calcular_clave_publica(g3, a3, p3)
B3 = calcular_clave_publica(g3, b3, p3)
s_alice3 = calcular_secreto_compartido(B3, a3, p3)
s_bob3 = calcular_secreto_compartido(A3, b3, p3)

print(f"\nClaves públicas: A={A3}, B={B3}")
print(f"Secreto compartido: {s_alice3}")
print(f"✓ Coinciden: {s_alice3 == s_bob3}")

---

# 📚 PARTE TEÓRICA

## 1. ¿Qué es Diffie-Hellman?

**Diffie-Hellman** es un protocolo criptográfico que permite a dos partes establecer un **secreto compartido** sobre un canal de comunicación público inseguro.

### Historia

- **1976**: Whitfield Diffie y Martin Hellman publican el protocolo
- **Revolución**: Primera solución práctica al problema de distribución de claves
- **Impacto**: Base de la criptografía de clave pública moderna
- **Premio Turing 2015**: Diffie y Hellman ganan el "Nobel de la computación"

### El Problema que Resuelve

**Problema tradicional:**
- Alice y Bob quieren comunicarse de forma segura
- Necesitan una clave compartida para cifrado simétrico
- ¿Cómo intercambiar la clave sin que un atacante la intercepte?

**Solución de Diffie-Hellman:**
- Alice y Bob pueden generar un secreto compartido
- Sin necesidad de intercambiar el secreto directamente
- Un atacante que observe todo el intercambio no puede calcular el secreto

### Analogía: La Mezcla de Colores

1. **Público**: Alice y Bob eligen un color base (amarillo)
2. **Privado**: Cada uno elige secretamente otro color
   - Alice elige rojo
   - Bob elige azul
3. **Mezcla privada**: Cada uno mezcla su color con el público
   - Alice: amarillo + rojo = naranja
   - Bob: amarillo + azul = verde
4. **Intercambio**: Se envían las mezclas (naranja ↔ verde)
5. **Secreto final**: Cada uno añade su color privado a la mezcla recibida
   - Alice: verde + rojo = marrón
   - Bob: naranja + azul = marrón
6. **Resultado**: ¡Ambos tienen el mismo color (marrón)!

Un atacante ve: amarillo, naranja y verde, pero no puede obtener el marrón.

## 2. Fundamento Matemático

### Componentes del Protocolo

**Parámetros públicos** (conocidos por todos, incluyendo atacantes):
- `p`: Número primo grande
- `g`: Generador (raíz primitiva módulo p)

**Claves privadas** (secretas, nunca compartidas):
- `a`: Clave privada de Alice (número aleatorio entre 2 y p-2)
- `b`: Clave privada de Bob (número aleatorio entre 2 y p-2)

**Claves públicas** (pueden compartirse públicamente):
- `A = g^a mod p`: Clave pública de Alice
- `B = g^b mod p`: Clave pública de Bob

**Secreto compartido:**
- Alice calcula: `s = B^a mod p = (g^b)^a mod p = g^(ba) mod p`
- Bob calcula: `s = A^b mod p = (g^a)^b mod p = g^(ab) mod p`

Como `ba = ab`, ambos obtienen el mismo secreto.

### ¿Por qué es Seguro?

Un atacante (Eve) que intercepta el canal ve:
- p, g (parámetros públicos)
- A (clave pública de Alice)
- B (clave pública de Bob)

Para calcular el secreto, Eve necesitaría encontrar `a` o `b`, lo que significa resolver:
```
g^a ≡ A (mod p)  →  a = log_g(A) mod p
```

Este es el **Problema del Logaritmo Discreto**, que es computacionalmente difícil para números suficientemente grandes.

## 3. El Problema del Logaritmo Discreto

### Definición

Dado:
- Primo `p`
- Generador `g`
- Valor `A = g^a mod p`

Encontrar `a` es el **Problema del Logaritmo Discreto (DLP)**.

### Comparación con Logaritmos Ordinarios

**Logaritmos reales** (fáciles):
```
2^10 = 1024  →  log₂(1024) = 10  ✓ Fácil de calcular
```

**Logaritmos discretos** (difíciles):
```
5^a ≡ 8 (mod 23)  →  a = ?  ✗ Muy difícil si p es grande
```

### Complejidad

Para un primo de n bits:
- **Dirección fácil** (exponenciación): O(n³) - tiempo polinomial
- **Dirección difícil** (logaritmo discreto): O(√p) - tiempo exponencial

**Ejemplo:**
- Para p de 2048 bits (≈ 10⁶¹⁷)
- Exponenciación: milisegundos
- Logaritmo discreto: ¡millones de años con computadoras actuales!

### Demostración

In [ ]:
import time

def logaritmo_discreto_fuerza_bruta(g, A, p):
    """
    Intenta encontrar 'a' tal que g^a ≡ A (mod p) por fuerza bruta.
    Solo para valores pequeños (demostración educativa).
    """
    for a_intento in range(1, p):
        if exponenciacion_rapida(g, a_intento, p) == A:
            return a_intento
    return None

# Demostración con valor pequeño
p_demo = 23
g_demo = 5
a_real = 6
A_demo = exponenciacion_rapida(g_demo, a_real, p_demo)

print("Demostración del Problema del Logaritmo Discreto")
print("="*60)
print(f"\nDado: p={p_demo}, g={g_demo}, A={A_demo}")
print(f"Encontrar: a tal que {g_demo}^a ≡ {A_demo} (mod {p_demo})")

inicio = time.time()
a_encontrado = logaritmo_discreto_fuerza_bruta(g_demo, A_demo, p_demo)
tiempo = time.time() - inicio

print(f"\nPor fuerza bruta: a = {a_encontrado}")
print(f"Tiempo: {tiempo*1000:.2f} ms")
print(f"\nPara p pequeño (23), probamos {p_demo-1} valores")
print(f"\nPara p grande (2048 bits ≈ 10^617):")
print(f"  Intentos necesarios: ≈ 10^617")
print(f"  Tiempo estimado: ¡mayor que la edad del universo!")
print(f"\n💡 Por eso Diffie-Hellman es seguro con primos grandes")

## 4. Flujo del Protocolo Paso a Paso

### Paso 0: Configuración (Una sola vez)

Se eligen parámetros públicos que todos usarán:
1. Elegir un primo grande `p`
2. Elegir un generador `g` módulo `p`
3. Publicar `p` y `g`

### Paso 1: Alice Genera su Par de Claves

1. Alice elige aleatoriamente `a` (privada)
2. Alice calcula `A = g^a mod p` (pública)
3. Alice guarda `a` en secreto
4. Alice publica `A`

### Paso 2: Bob Genera su Par de Claves

1. Bob elige aleatoriamente `b` (privada)
2. Bob calcula `B = g^b mod p` (pública)
3. Bob guarda `b` en secreto
4. Bob publica `B`

### Paso 3: Intercambio Público

Alice y Bob intercambian sus claves públicas sobre un canal inseguro:
- Alice envía `A` a Bob
- Bob envía `B` a Alice

⚠️ Un atacante puede ver `A` y `B`, pero no `a` ni `b`

### Paso 4: Cálculo del Secreto

**Alice calcula:**
```
s = B^a mod p
```

**Bob calcula:**
```
s = A^b mod p
```

**Resultado:** Ambos obtienen el mismo valor `s`

### Paso 5: Uso del Secreto

Alice y Bob usan `s` como:
- Clave para cifrado simétrico (AES, etc.)
- Material para derivar otras claves
- Semilla para generadores aleatorios

## 5. Seguridad y Vulnerabilidades

### Fortalezas

1. **Problema matemático difícil**: Basado en el logaritmo discreto
2. **No requiere canal seguro**: Solo necesita autenticación
3. **Perfect Forward Secrecy**: Si las claves privadas se comprometen después, sesiones pasadas siguen seguras
4. **Eficiente**: Cálculos rápidos incluso con números grandes

### Vulnerabilidades

1. **Ataque Man-in-the-Middle (MITM)**:
   - Sin autenticación, un atacante puede interceptar
   - Eve se hace pasar por Bob ante Alice y por Alice ante Bob
   - Solución: Usar certificados digitales o firmas

2. **Parámetros débiles**:
   - Primos pequeños son vulnerables
   - Generadores mal elegidos pueden ser inseguros
   - Solución: Usar parámetros estándar (NIST, RFC)

3. **Ataques a la implementación**:
   - Timing attacks
   - Side-channel attacks
   - Números aleatorios débiles

4. **Computadoras cuánticas**:
   - Algoritmo de Shor puede romper DLP
   - Solución futura: Criptografía post-cuántica

### Tamaños Recomendados (2024)

| Uso | Tamaño de p | Seguridad equivalente |
|-----|-------------|----------------------|
| **Mínimo** | 2048 bits | 112 bits |
| **Recomendado** | 3072 bits | 128 bits |
| **Alto** | 4096+ bits | 152+ bits |

## 6. Aplicaciones Modernas

Diffie-Hellman es fundamental en:

### TLS/SSL (HTTPS)
- Establece claves de sesión para navegación web segura
- Variante: Elliptic Curve Diffie-Hellman (ECDH)

### VPNs
- IPsec usa Diffie-Hellman para establecer túneles seguros
- IKE (Internet Key Exchange) protocol

### SSH
- Protocolo de shell seguro
- Diffie-Hellman para el intercambio inicial de claves

### Signal Protocol
- Mensajería segura (WhatsApp, Signal, etc.)
- Usa variantes de Diffie-Hellman

### Bitcoin/Blockchain
- ECDH para generar direcciones
- Transacciones confidenciales

---

## 📊 Suite de Pruebas Completa

In [ ]:
def suite_pruebas_diffie_hellman():
    """Suite completa de pruebas."""
    
    print("="*80)
    print(" "*25 + "SUITE DE PRUEBAS")
    print("="*80)
    
    casos = [
        {"nombre": "Caso 1", "p": 23, "g": 5, "a": 6, "b": 15},
        {"nombre": "Caso 2", "p": 47, "g": 7, "a": 12, "b": 25},
        {"nombre": "Caso 3", "p": 97, "g": 3, "a": 42, "b": 57},
        {"nombre": "Caso 4", "p": 199, "g": 11, "a": 88, "b": 123},
        {"nombre": "Caso 5", "p": 997, "g": 7, "a": 456, "b": 789},
    ]
    
    exitos = 0
    fallos = 0
    
    for caso in casos:
        print(f"\n{caso['nombre']}: p={caso['p']}, g={caso['g']}")
        print("-"*80)
        
        try:
            p, g, a, b = caso['p'], caso['g'], caso['a'], caso['b']
            
            A = calcular_clave_publica(g, a, p)
            B = calcular_clave_publica(g, b, p)
            
            s_alice = calcular_secreto_compartido(B, a, p)
            s_bob = calcular_secreto_compartido(A, b, p)
            
            print(f"Claves públicas: A={A}, B={B}")
            print(f"Secretos: Alice={s_alice}, Bob={s_bob}")
            
            if s_alice == s_bob:
                print("✅ ÉXITO - Secretos coinciden")
                exitos += 1
            else:
                print("❌ FALLO - Secretos NO coinciden")
                fallos += 1
        
        except Exception as e:
            print(f"❌ ERROR: {e}")
            fallos += 1
    
    print("\n" + "="*80)
    print(" "*32 + "RESUMEN")
    print("="*80)
    print(f"Total: {len(casos)}")
    print(f"✅ Éxitos: {exitos}")
    print(f"❌ Fallos: {fallos}")
    print(f"Tasa de éxito: {(exitos/len(casos)*100):.1f}%")

suite_pruebas_diffie_hellman()

---

## 🎓 Conclusiones y Referencias

### Lo que aprendimos

1. **Diffie-Hellman**: Protocolo revolucionario para intercambio de claves
2. **Logaritmo discreto**: Problema matemático que garantiza la seguridad
3. **Exponenciación modular**: Operación fundamental eficiente
4. **Criptografía asimétrica**: Base de la seguridad moderna

### Importancia Histórica

Diffie-Hellman fue **revolucionario** porque:
- Primer protocolo práctico de clave pública (1976)
- Resolvió el problema de distribución de claves
- Inspiró RSA y otros sistemas
- Base de Internet seguro (HTTPS, VPN, etc.)

### Referencias

- **Diffie, W. & Hellman, M.** (1976): "New Directions in Cryptography"
- **RFC 2631**: Diffie-Hellman Key Agreement Method
- **RFC 3526**: More Modular Exponential (MODP) Diffie-Hellman groups
- **NIST SP 800-56A**: Recommendation for Pair-Wise Key Establishment Schemes

---

**Laboratorio completado exitosamente** ✅